# Hugging Face lookup checks

Use this notebook to inspect the Hugging Face lookup path without model-weight downloads. It covers newly created Hub models, models in a chosen family, and later model releases in that family.

The first three checks use `enrich=False`, which reads Hub listing metadata only. Run the final optional cell to fetch config and tokenizer metadata for a small, explicit shortlist.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    raise RuntimeError('Start this notebook from the repository root.')
sys.path.insert(0, str(ROOT / 'src'))

from hf_model_watch.hub.lookup import lookup_models
from hf_model_watch.recommender.discovery import _model_release_version

ARTIFACT_DIR = ROOT / 'artifacts' / 'lookup_checks'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REQUESTED_MODEL = 'Qwen/Qwen3-4B'
FAMILY = 'Qwen'
LIMIT = 25

def compact(records):
    return [
        {key: record.get(key) for key in (
            'model_id', 'created_at', 'last_modified', 'downloads',
            'estimated_params_billion', 'license', 'pipeline_tag',
        )}
        for record in records
    ]


## 1. Newly created Hugging Face models

This asks the lookup tool for `text-generation` repositories ordered by repository creation time.

In [ ]:
new_models = lookup_models(
    query='text-generation',
    sort='createdAt',
    limit=LIMIT,
    enrich=False,
    output_path=ARTIFACT_DIR / 'new_models.jsonl',
)
compact(new_models)


## 2. Recently created repositories in a chosen model family

In [ ]:
family_models = lookup_models(
    family=FAMILY,
    sort='createdAt',
    limit=LIMIT,
    enrich=False,
    output_path=ARTIFACT_DIR / 'family_models.jsonl',
)
compact(family_models)


## 3. Later releases at a similar parameter scale

A later release must have a greater generation version in its repository ID (for example, `Qwen3.5` after `Qwen3`) and be between half and twice the requested model's parameter count.

In [ ]:
requested = lookup_models(
    model_id=REQUESTED_MODEL,
    output_path=ARTIFACT_DIR / 'requested_model.jsonl',
)[0]
target_size = requested.get('estimated_params_billion')
requested_release = _model_release_version(REQUESTED_MODEL, FAMILY)

later_releases = [
    record for record in family_models
    if (
        record['model_id'] != REQUESTED_MODEL
        and (_model_release_version(record['model_id'], FAMILY) or ()) > requested_release
        and record.get('estimated_params_billion') is not None
        and target_size / 2 <= record['estimated_params_billion'] <= target_size * 2
    )
]
compact(later_releases)


## 4. Optional: hydrate only the shortlisted repositories

This is the only step that fetches `config.json` and `tokenizer_config.json`. It limits that work to the first two later-release candidates.

In [ ]:
shortlist = later_releases[:2]
hydrated = [
    lookup_models(
        model_id=record['model_id'],
        output_path=ARTIFACT_DIR / f"{record['model_id'].replace('/', '_')}.jsonl",
    )[0]
    for record in shortlist
]
compact(hydrated)
